# Pipeline Prediksi Log Return Kurs USD/IDR dengan Sentimen Berita (Fitur = Data Labelling Saja)

Versi ini **tanpa feature engineering**. Satu-satunya fitur model adalah hasil data labelling
(VADER dan Loughran-McDonald) yang diagregasi per hari trading. Tidak ada fitur kurs (lag/rolling/momentum),
fitur waktu, fitur jumlah berita, maupun TF-IDF/SVD.

Urutan pipeline:
1. Preprocessing (berita & kurs)
2. Data labelling: **VADER** dan **Loughran-McDonald (LM)**
3. Penyelarasan berita ke hari trading + agregasi harian + merge dengan kurs + target
4. Split kronologis **70 / 15 / 15**
5. Dataset final (fitur = kolom hasil labelling)
6. Normalisasi (StandardScaler, **fit di train saja**) + training semua model regresi dengan **parameter default**
7. Ablation study antar metode labelling (MAE, MSE, RMSE, DirAcc)
8. Feature selection top-K dan perbandingannya


# 1. Preprocessing

In [ ]:
import re
import html
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import nltk
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

In [ ]:
news_raw = pd.read_csv('data/raw/cnbc_with_published.csv')
bi_raw = pd.read_csv('data/raw/bi-usd-rate.csv')

print("Berita geopolitik :", news_raw.shape)
print("Kurs USD BI       :", bi_raw.shape)
news_raw.head(5)


In [ ]:
news_raw.info()

In [ ]:
# Parse timestamp mentah GEO dan pisahkan tanggal & jam dengan timezone WIB.
news_raw["date_raw_dt"] = pd.to_datetime(news_raw["date_raw"], errors="coerce")
news_raw["date_raw_dt_wib"] = (
    pd.to_datetime(news_raw["date_raw_dt"], errors="coerce", utc=True)
    .dt.tz_convert("Asia/Jakarta")
    .dt.tz_localize(None)
)
news_raw["tanggal"] = news_raw["date_raw_dt_wib"].dt.strftime("%m/%d/%Y")
news_raw["jam"] = news_raw["date_raw_dt_wib"].dt.strftime("%I:%M %p")
news_raw = news_raw.drop(columns=["date_raw"], errors="ignore")

print("Kolom Berita Geopolitik:")
news_raw[["title", "tanggal", "jam", "date_raw_dt_wib"]].head(5)


In [ ]:
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")
bi_raw["tanggal"] = bi_raw["Tanggal_dt"].dt.strftime("%m/%d/%Y")
bi_raw["jam"] = bi_raw["Tanggal_dt"].dt.strftime("%I:%M %p")
bi_raw = bi_raw.drop(
    columns=["Tanggal"],
    errors="ignore"
)

print("Kolom Kurs BI:")
bi_raw.head(5)

## 1.1 Cleaning berita

In [ ]:
#delete duplicate rows
def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

news = news_raw.copy()
# Ukuran data awal
print(f"Baris awal                         : {len(news_raw)}")

# Filter rentang tanggal inklusif: 1 September 2021 - 1 September 2026
start_date = pd.Timestamp("2021-09-01")
end_date = pd.Timestamp("2026-09-01 23:59:59")
news = news[news["date_raw_dt"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang 1 September 2021 - 1 September 2026 : {len(news)}")

# Setelah di drop URL
news = news.drop_duplicates(subset=["url"], keep="first")
print(f"Setelah dedup url                  : {len(news)}")

# Setelah di drop title + date_raw
news = news.drop_duplicates(subset=["title", "tanggal", "jam"], keep="first")
print(f"Setelah dedup title+date_raw       : {len(news)}")

In [ ]:
# Filter full_text yang kosong

# Buang baris yang tidak memiliki isi teks pada ketiga sumber konten.
text_columns = ["title", "preview_summary", "full_text"]
text_content = news[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
mask_empty = text_content.eq("")
news = news.loc[~mask_empty].copy()
print(f"Setelah drop teks kosong             : {len(news)}")

# Buang item multimedia berdasarkan URL, judul, dan section.
non_article_pattern = r"\b(video|videos|image|images|photo|photos|foto|gambar|audio|podcast|gallery|galeri|live)\b|\.(mp4|mov|avi|mp3|wav|jpg|jpeg|png)(?:$|[?#])"
content_metadata = news[["url", "title", "section"]].fillna("").astype(str).agg(" ".join, axis=1)
mask_non_article = content_metadata.str.contains(non_article_pattern, case=False, regex=True, na=False)
news = news.loc[~mask_non_article].copy()
print(f"Setelah drop video/gambar/audio      : {len(news)}")

news.head(5)

In [ ]:
# Panjang text minimal 350

effective_text = news["full_text"].fillna(news["preview_summary"])
news = news.loc[effective_text.apply(word_count) >= 350]
print(f"Setelah filter panjang teks min     : {len(news)}")

In [ ]:
#Filter keyword yang relevan dengan geopolitik dan ekonomi
RELEVANCE_KEYWORDS = [
    "geopolitics",
    "geopolitical risk",
    "geopolitical tensions",
    "geopolitical fragmentation",
    "armed conflict",
    "global conflict",
    "international conflict",
    "military tensions",
    "national security",
    "sanctions",
    "trade war",
    "tariffs",
    "embargo",
    "export controls",
    "protectionism",
    "supply chain disruption",
    "diplomacy",
    "foreign policy",
    "nato",
    "brics",
    "opec",
    "g7",
]

EXCLUDE_SECTIONS = [
    "sports", "entertainment", "television", "lifestyle", "travel",
    "food retail", "beer, wine & spirits", "college", "modern medicine",
    "life changes", "fashion", "celebrity",
]

def is_relevant(row):
    """Berita dianggap relevan jika keyword_matched terisi ATAU teks memuat
    salah satu kata kunci geopolitik/ekonomi pada RELEVANCE_KEYWORDS."""
    if isinstance(row["keyword_matched"], str) and row["keyword_matched"].strip():
        return True
    haystack = f"{row.get('title', '')} {row.get('preview_summary', '')}".lower()
    return any(kw in haystack for kw in RELEVANCE_KEYWORDS)

mask_relevant = news.apply(is_relevant, axis=1)
news = news.loc[mask_relevant]

print(f"Setelah filter relevansi kata kunci : {len(news)}")

# 4.5 Filter kategori/section yang tidak relevan
section_lower = news["section"].fillna("").str.lower()
mask_section = ~section_lower.isin(EXCLUDE_SECTIONS)
news = news.loc[mask_section].reset_index(drop=True)

print(f"Setelah filter kategori/section     : {len(news)}")

news.head(3)


In [ ]:
# delete missing values
jumlah_nan = news["full_text"].isna().sum()

print("Jumlah full_text NaN:", jumlah_nan)

news_nan = news.loc[news["full_text"].isna()]

news_nan[
    ["url", "title", "preview_summary", "full_text"]
].head()

news = news.dropna(subset=["full_text"]).reset_index(drop=True)

print("Jumlah baris setelah drop NaN:", len(news))
print("Sisa NaN pada full_text:", news["full_text"].isna().sum())

In [ ]:
#menggabungkan title, preview, dan full_text

# Tidak ada nilai kosong yang mengganggu penggabungan teks.
news[["title", "preview_summary", "full_text"]] = (
    news[["title", "preview_summary", "full_text"]].fillna("")
)

# Gabungkan title + preview_summary + full_text.
news["combined_raw"] = (
    ((news["title"] + " ") * 2)
    + news["preview_summary"] + " "
    + news["full_text"]
)

print(f"Jumlah baris: {len(news)}")
news[["title", "combined_raw"]].head(3)

In [ ]:
import unicodedata
from collections import Counter

symbol_counter = Counter()
for text in news["combined_raw"]:
    for ch in text:
        if not (ch.isalnum() or ch.isspace()):
            symbol_counter[ch] += 1

symbol_df = pd.DataFrame(
    [
        (ch, unicodedata.name(ch, "UNKNOWN"), count)
        for ch, count in symbol_counter.most_common()
    ],
    columns=["simbol", "nama_unicode", "jumlah"],
)

print(f"Jumlah jenis simbol unik pada combined_raw: {len(symbol_df)}")
symbol_df

In [ ]:
#cleaning dari tanda baca yang tidak berguna

URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)

# Hanya simbol mata uang yang diubah jadi kata.
# Simbol lain (&, +, tanda baca umum) dibuang di NON_ALNUM_RE.
CURRENCY_WORDS = {
    "%": "percent",
    "$": "dollar",
    "£": "pound",
    "€": "euro",
    "¥": "yen",
}
CURRENCY_RE = re.compile("|".join(re.escape(sym) for sym in CURRENCY_WORDS))
NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9\s]")  # simbol/tanda baca sisa dibuang; huruf & angka dipertahankan
MULTI_SPACE_RE = re.compile(r"\s+")


def normalize_text(text: str) -> str:
    """Hapus HTML/URL/email/emoji, ubah simbol mata uang jadi kata, angka dipertahankan."""
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = HTML_TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = CURRENCY_RE.sub(lambda m: f" {CURRENCY_WORDS[m.group(0)]} ", t)
    t = NON_ALNUM_RE.sub(" ", t)
    t = MULTI_SPACE_RE.sub(" ", t).strip()
    return t


news["normalized_text"] = news["combined_raw"].apply(normalize_text)

news[["combined_raw", "normalized_text"]].head(3)

In [ ]:
# lowercasing
news["lower_text"] = news["normalized_text"].str.lower()

news[["normalized_text", "lower_text"]].head(3)

In [ ]:
#memastikan bahasanya english semua
def detect_language(text: str) -> str:
    if not isinstance(text, str) or len(text.strip()) < 20:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


news["detected_lang"] = news["lower_text"].apply(detect_language)

print("Distribusi bahasa terdeteksi:")
print(news["detected_lang"].value_counts().head(10))

baris_sebelum = len(news)
news = news.loc[news["detected_lang"] == "en"].reset_index(drop=True)
print(f"\nSetelah filter bahasa Inggris (en) : {len(news)} (dibuang {baris_sebelum - len(news)} baris)")

news[["lower_text", "detected_lang"]].head(3)

## 1.2 Cleaning data kurs

In [ ]:
# Gabungkan tanggal dan jam untuk satu kolom datetime kanonik.
bi_raw["Tanggal"] = pd.to_datetime(
    bi_raw["tanggal"].astype(str).str.strip() + " " + bi_raw["jam"].astype(str).str.strip(),
    format="%m/%d/%Y %I:%M %p",
    errors="coerce",
)

print(bi_raw.shape)
bi_raw.head(3)

In [ ]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Hapus pemisah ribuan dan konversi nilai ke float."""
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")



bi_raw["Kurs Jual"] = clean_numeric(bi_raw["Kurs Jual"])
bi_raw["Kurs Beli"] = clean_numeric(bi_raw["Kurs Beli"])
bi_raw["NO"] = pd.to_numeric(bi_raw["NO"], errors="coerce")
bi_raw["Nilai"] = pd.to_numeric(bi_raw["Nilai"], errors="coerce")

print(f"Baris awal BI                 : {len(bi_raw)}")

# Pertahankan hanya rentang 1 September 2021 sampai 1 September 2026.
bi = bi_raw.loc[bi_raw["Tanggal"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang tanggal: {len(bi)}")

mask_valid_numeric = (
    bi["Kurs Jual"].notna()
    & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0)
    & (bi["Kurs Beli"] > 0)
)
bi = bi.loc[mask_valid_numeric]
print(f"Setelah filter numerik valid  : {len(bi)}")

mask_valid_spread = bi["Kurs Jual"] > bi["Kurs Beli"]
bi = bi.loc[mask_valid_spread]
print(f"Setelah filter Jual > Beli    : {len(bi)}")

bi = (
    bi.sort_values("Tanggal")
    .drop_duplicates(subset=["Tanggal"], keep="last")
    .reset_index(drop=True)
)
print(f"Setelah dedup tanggal         : {len(bi)}")

bi.head(3)

# 2. Data Labelling (VADER & LM)

## 2.1 VADER

In [ ]:
USE_VADER = True   # <- True = aktif, False = nonaktif (kolom VADER dibuang dari news)

# Hitung VADER tiap judul (per artikel, teks mentah tanpa tokenisasi/lowercase/lemmatisasi)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon', quiet=True)
vader = SentimentIntensityAnalyzer()

def score_title(t):
    # judul kosong jangan dihitung sebagai 0 (netral), biarkan NaN
    if isinstance(t, str) and t.strip():
        return vader.polarity_scores(t)['compound']
    return np.nan

news = news.drop(columns=['vader_title'], errors='ignore')   # bersihkan sisa run sebelumnya

if USE_VADER:
    news['vader_title'] = news['title'].apply(score_title)
    print('VADER aktif')
    display(news[['title', 'vader_title']].head())
else:
    print('VADER dinonaktifkan')

## 2.2 Loughran-McDonald

In [ ]:
# ===== LOUGHRAN-McDONALD: muat kamus & fungsi skor (jalankan sekali) =====
# Kategori LM yang dipakai. Boleh ditambah: 'constraining', 'strong_modal', 'weak_modal'.
# 'negative' dan 'positive' wajib ada (dipakai untuk skor net = positive - negative).
LM_CATS = ['negative', 'positive', 'uncertainty', 'litigious']

LM_COLS = {'negative': 'Negative', 'positive': 'Positive', 'uncertainty': 'Uncertainty',
           'litigious': 'Litigious', 'constraining': 'Constraining',
           'strong_modal': 'Strong_Modal', 'weak_modal': 'Weak_Modal'}
assert {'negative', 'positive'} <= set(LM_CATS), "LM_CATS harus memuat 'negative' dan 'positive'"

# Kamus resmi: Loughran-McDonald Master Dictionary 1993-2025 (sraf.nd.edu), letakkan di data/lexicon/
lm_files = sorted(Path('data/lexicon').glob('Loughran*McDonald*MasterDictionary*.csv'))
if not lm_files:
    raise FileNotFoundError("Taruh Loughran-McDonald_MasterDictionary_1993-2025.csv di data/lexicon/")
lm_path = lm_files[-1]

lm_df = pd.read_csv(lm_path, usecols=['Word'] + [LM_COLS[c] for c in LM_CATS])
lm_df = lm_df.dropna(subset=['Word'])                       # buang baris tanpa kata
lm_df['Word'] = lm_df['Word'].astype(str).str.lower()
LM_SETS = {c: set(lm_df.loc[lm_df[LM_COLS[c]] > 0, 'Word']) for c in LM_CATS}
print(f"Kamus LM dari: {lm_path}")
print({c: len(s) for c, s in LM_SETS.items()})

WORD_RE = re.compile(r'[a-z]+')

def lm_scores(text):
    """Proporsi kata tiap kategori LM terhadap total kata (text harus sudah lowercase)."""
    tokens = WORD_RE.findall(text) if isinstance(text, str) else []
    n = len(tokens)
    if n == 0:
        return [np.nan] * len(LM_CATS)            # teks kosong -> NaN, bukan 0
    return [sum(t in LM_SETS[c] for t in tokens) / n for c in LM_CATS]

In [ ]:
USE_LM_TITLE = True   # <- True = aktif, False = nonaktif (kolom lm_title_* dibuang dari news)

news = news.drop(columns=[c for c in news.columns if c.startswith('lm_title_')])   # bersihkan sisa run sebelumnya

if USE_LM_TITLE:
    assert LM_SETS is not None, 'Kamus LM belum termuat (lihat cell setup LM di atas)'
    scores = news['title'].apply(lambda t: lm_scores(normalize_text(t).lower())).tolist()
    lm_t = pd.DataFrame(scores, columns=[f'lm_title_{c}' for c in LM_CATS], index=news.index)
    lm_t['lm_title_net'] = lm_t['lm_title_positive'] - lm_t['lm_title_negative']
    for col in lm_t.columns:
        news[col] = lm_t[col].values
    print('LM judul aktif')
    display(news[['title'] + list(lm_t.columns)].head())
else:
    print('LM judul dinonaktifkan')

In [ ]:
USE_LM_TEXT = True   # <- True = aktif, False = nonaktif (kolom lm_text_* dibuang dari news)

news = news.drop(columns=[c for c in news.columns if c.startswith('lm_text_')])   # bersihkan sisa run sebelumnya

if USE_LM_TEXT:
    assert LM_SETS is not None, 'Kamus LM belum termuat (lihat cell setup LM di atas)'
    # isi berita (full_text) dinormalisasi seperti pipeline lain, lalu lowercase.
    # Tanpa stopword removal/lemmatisasi: kamus LM sudah memuat bentuk turunan kata.
    scores = news['full_text'].apply(lambda t: lm_scores(normalize_text(t).lower())).tolist()
    lm_x = pd.DataFrame(scores, columns=[f'lm_text_{c}' for c in LM_CATS], index=news.index)
    lm_x['lm_text_net'] = lm_x['lm_text_positive'] - lm_x['lm_text_negative']
    for col in lm_x.columns:
        news[col] = lm_x[col].values
    print('LM isi aktif')
    display(news[list(lm_x.columns)].describe().T)
else:
    print('LM isi dinonaktifkan')

# 3. Penyelarasan, Agregasi Harian & Target

## 3.1 Penyelarasan berita ke hari trading

In [ ]:
#change timezone to WIB
news['published_raw'] = pd.to_datetime(news['published_raw'], errors='coerce', utc=True)
news['published_raw'] = news['published_raw'].dt.tz_convert('Asia/Jakarta')

#pisahin tanggal & waktu
news['date'] = news['published_raw'].dt.date
news['time'] = news['published_raw'].dt.time

In [ ]:
def align_to_trading_date(dt):
    # Step 1: kalau lewat cutoff 15:00:00, geser ke hari berikutnya
    cutoff = dt.replace(hour=15, minute=0, second=0, microsecond=0)
    if dt > cutoff:
        dt = dt + pd.Timedelta(days=1)

    # Step 2: kalau hasilnya jatuh di weekend, geser ke Senin
    if dt.weekday() == 5:      # Sabtu
        dt = dt + pd.Timedelta(days=2)
    elif dt.weekday() == 6:    # Minggu
        dt = dt + pd.Timedelta(days=1)

    return dt.normalize()  # buang jam, sisakan tanggal saja

news['aligned_trading_date'] = news['date_raw_dt'].apply(align_to_trading_date)

In [ ]:
# Snap ke hari trading yang ada di data kurs (jalankan SETELAH cell align_to_trading_date di atas)
td = pd.DatetimeIndex(bi_raw['Tanggal_dt']).normalize().unique().sort_values()   # hari trading yang ada di data kurs
ad = pd.DatetimeIndex(news['aligned_trading_date'])

pos = td.searchsorted(ad, side='left')       # hari trading pertama yang >= tanggal hasil alignment
ok = pos < len(td)                           # artikel setelah hari kurs terakhir tidak punya tujuan
print(f"Artikel dibuang karena melewati hari kurs terakhir: {(~ok).sum()}")

news = news.loc[ok].copy()
news['aligned_trading_date'] = td[pos[ok]]

## 3.2 Agregasi harian & merge dengan kurs

In [ ]:
# urutkan dulu supaya urutan artikel konsisten
news_sorted = news.sort_values('date_raw_dt')

# news_count hanya untuk referensi (membedakan hari tanpa berita), BUKAN fitur model
agg_spec = {'news_count': ('title', 'size')}

# statistik VADER untuk 1 tanggal aligned
if 'vader_title' in news_sorted.columns:
    agg_spec.update({
        'vader_title_mean':      ('vader_title', 'mean'),
        'vader_title_min':       ('vader_title', 'min'),
        'vader_title_max':       ('vader_title', 'max'),
        'vader_title_std':       ('vader_title', 'std'),
        'vader_title_pos_ratio': ('vader_title', lambda x: (x.dropna() > 0.05).mean()),   # ambang standar VADER
        'vader_title_neg_ratio': ('vader_title', lambda x: (x.dropna() < -0.05).mean()),
    })

# LM (judul/isi): rata-rata harian dari skor per artikel (hanya kolom yang aktif)
sent_cols = [c for c in news_sorted.columns if c.startswith(('lm_title_', 'lm_text_'))]
for c in sent_cols:
    agg_spec[c] = (c, 'mean')

daily_news = (
    news_sorted.groupby('aligned_trading_date')
               .agg(**agg_spec)
               .reset_index()
)

# 1 artikel saja -> std NaN, ganti 0
if 'vader_title_std' in daily_news.columns:
    daily_news['vader_title_std'] = daily_news['vader_title_std'].fillna(0.0)

daily_news

In [ ]:
# 1. Samakan tipe & buang jam di kedua kunci
bi_raw['Tanggal_dt'] = pd.to_datetime(bi_raw['Tanggal_dt']).dt.normalize()
daily_news['aligned_trading_date'] = pd.to_datetime(daily_news['aligned_trading_date']).dt.normalize()

# 2. Cek duplikat kunci (kalau ada, baris akan membengkak setelah merge)
print(bi_raw['Tanggal_dt'].duplicated().sum())
print(daily_news['aligned_trading_date'].duplicated().sum())

# 3. Merge
merged_data = pd.merge(
    bi_raw,
    daily_news,
    left_on='Tanggal_dt',
    right_on='aligned_trading_date',
    how='left',
    validate='one_to_one'   # error kalau ada duplikat, jadi ketahuan
).drop(columns='aligned_trading_date')

# 4. Hari tanpa berita: news_count = 0 dan skor sentimen netral (0.0)
merged_data['news_count'] = merged_data['news_count'].fillna(0).astype(int)
sent_cols = [c for c in merged_data.columns if c.startswith(('vader_title_', 'lm_title_', 'lm_text_'))]
merged_data[sent_cols] = merged_data[sent_cols].fillna(0.0)

merged_data

In [ ]:
# Simpan hanya kolom yang dibutuhkan: kurs (untuk target), news_count (referensi), dan kolom hasil labelling
label_cols_all = [c for c in merged_data.columns if c.startswith(('vader_title_', 'lm_title_', 'lm_text_'))]
merged_data_clean = merged_data[['Tanggal_dt', 'Kurs Jual', 'Kurs Beli', 'news_count'] + label_cols_all].copy()
merged_data_clean

## 3.3 Target (bukan feature engineering)
Tidak ada fitur turunan. Fitur model = kolom hasil labelling pada hari *t* apa adanya (tanpa lag, rolling, atau diff).
Target = log return hari trading berikutnya (t -> t+1). Kolom `kurs_mid` dan `ret` hanya disimpan sebagai **referensi**
(untuk baseline naive "Return kemarin"), tidak dimasukkan ke model.


In [ ]:
d = merged_data_clean.sort_values('Tanggal_dt', ascending=True).reset_index(drop=True)

# --- Target & kolom referensi (BUKAN fitur model) ---
d['kurs_mid'] = (d['Kurs Jual'] + d['Kurs Beli']) / 2
log_mid = np.log(d['kurs_mid'])
d['ret'] = log_mid.diff()              # log return hari ini (t-1 -> t), hanya untuk baseline naive
d['target'] = d['ret'].shift(-1)       # log return hari berikutnya (t -> t+1)

# --- Fitur = hasil labelling apa adanya, dikelompokkan per metode ---
LABEL_GROUPS = {
    'VADER':    [c for c in d.columns if c.startswith('vader_')],
    'LM judul': [c for c in d.columns if c.startswith('lm_title_')],
    'LM isi':   [c for c in d.columns if c.startswith('lm_text_')],
}
LABEL_GROUPS = {g: cols for g, cols in LABEL_GROUPS.items() if cols}
feat_label = [c for cols in LABEL_GROUPS.values() for c in cols]

# Buang baris pertama (ret NaN) dan baris terakhir (target NaN), SEBELUM split
merged_data_clean = d.dropna(subset=['ret', 'target'] + feat_label).reset_index(drop=True)

print(merged_data_clean.shape)
print({g: len(c) for g, c in LABEL_GROUPS.items()}, '| total fitur:', len(feat_label))
merged_data_clean[['Tanggal_dt', 'kurs_mid', 'ret', 'target'] + feat_label[:4]].head()

# 4. Split Data (70 / 15 / 15, kronologis)

In [ ]:
merged_data_clean = merged_data_clean.sort_values('Tanggal_dt', ascending=True).reset_index(drop=True)

n = len(merged_data_clean)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train = merged_data_clean.iloc[:train_end].copy()
val   = merged_data_clean.iloc[train_end:val_end].copy()
test  = merged_data_clean.iloc[val_end:].copy()

# buang 1 baris terakhir di train & val: target baris itu (t+1) adalah hari pertama set berikutnya
train = train.iloc[:-1].copy()
val   = val.iloc[:-1].copy()

assert train['Tanggal_dt'].max() < val['Tanggal_dt'].min() < test['Tanggal_dt'].min()

tot = len(train) + len(val) + len(test)
for name, d_ in [('Train', train), ('Val', val), ('Test', test)]:
    print(f"{name:5}: {len(d_):5d} ({len(d_) / tot:.1%}) | {d_['Tanggal_dt'].min().date()} -> {d_['Tanggal_dt'].max().date()}")

# 5. Dataset Final (fitur = hasil labelling saja)

In [ ]:
# ===== DATAFRAME FINAL: kolom hasil labelling + target (+ kolom referensi kurs_mid, ret) =====
final_cols = ['Tanggal_dt', 'kurs_mid', 'ret'] + feat_label + ['target']

final_train = train[final_cols].reset_index(drop=True)
final_val   = val[final_cols].reset_index(drop=True)
final_test  = test[final_cols].reset_index(drop=True)

for name, d_ in [('Train', final_train), ('Val', final_val), ('Test', final_test)]:
    assert d_.isna().sum().sum() == 0, f'{name} masih ada NaN'
    print(f"{name:5}: {d_.shape}")

# peta fitur -> grup (dipakai untuk ringkasan komposisi fitur)
GROUP_OF = {c: g for g, cols in LABEL_GROUPS.items() for c in cols}

# simpan
Path('data/processed').mkdir(parents=True, exist_ok=True)
for name, d_ in [('train', final_train), ('val', final_val), ('test', final_test)]:
    d_.to_csv(f'data/processed/{name}.csv', index=False, date_format='%Y-%m-%d')

# 6. Setup Modelling
- Normalisasi: `StandardScaler` di-fit **hanya di train** untuk tiap set fitur, lalu dipakai untuk val/test.
- Semua model memakai **parameter default** library (tanpa tuning). `random_state` hanya untuk reproduksibilitas.
- Metrik: MAE, MSE, RMSE, DirAcc (arah naik/turun benar, baris dengan target tepat 0 tidak dihitung).
- `PredStd` ≈ 0 berarti model menghasilkan prediksi konstan (umum pada Lasso/ElasticNet/SVR default karena target sangat kecil).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              AdaBoostRegressor, GradientBoostingRegressor)

pd.set_option('display.float_format', lambda x: f'{x:.6g}')
SEED = 42

# Semua model regresi: PARAMETER DEFAULT (tanpa tuning)
MODELS = {
    'LinearRegression': lambda: LinearRegression(),
    'Ridge':            lambda: Ridge(),
    'Lasso':            lambda: Lasso(),
    'ElasticNet':       lambda: ElasticNet(),
    'BayesianRidge':    lambda: BayesianRidge(),
    'KNN':              lambda: KNeighborsRegressor(),
    'SVR':              lambda: SVR(),
    'DecisionTree':     lambda: DecisionTreeRegressor(random_state=SEED),
    'RandomForest':     lambda: RandomForestRegressor(random_state=SEED, n_jobs=-1),
    'ExtraTrees':       lambda: ExtraTreesRegressor(random_state=SEED, n_jobs=-1),
    'AdaBoost':         lambda: AdaBoostRegressor(random_state=SEED),
    'GradientBoosting': lambda: GradientBoostingRegressor(random_state=SEED),
}
try:
    from xgboost import XGBRegressor
    MODELS['XGBoost'] = lambda: XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0)
except ImportError:
    print('xgboost tidak terpasang -> dilewati')
try:
    from lightgbm import LGBMRegressor
    MODELS['LightGBM'] = lambda: LGBMRegressor(random_state=SEED, n_jobs=-1, verbosity=-1)
except ImportError:
    print('lightgbm tidak terpasang -> dilewati')

METRICS = ['MAE', 'MSE', 'RMSE', 'DirAcc']

def evaluate(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    mse = mean_squared_error(y_true, y_pred)
    nz = y_true != 0                                   # arah tidak terdefinisi kalau target tepat 0
    dir_acc = np.mean(np.sign(y_pred[nz]) == np.sign(y_true[nz])) if nz.any() else np.nan
    return {'MAE': mean_absolute_error(y_true, y_pred), 'MSE': mse, 'RMSE': np.sqrt(mse), 'DirAcc': dir_acc}

y_tr, y_va, y_te = final_train['target'], final_val['target'], final_test['target']

def prepare(cols):
    """Normalisasi: scaler di-fit HANYA di train, lalu diterapkan ke val & test."""
    scaler = StandardScaler().fit(final_train[cols])
    return (scaler.transform(final_train[cols]),
            scaler.transform(final_val[cols]),
            scaler.transform(final_test[cols]))

def fit_eval(make_model, X_tr, X_va, X_te):
    """Latih model (parameter default) di train, evaluasi di val & test."""
    model = make_model().fit(X_tr, y_tr)
    out = {}
    for split, X, y in (('Val', X_va, y_va), ('Test', X_te, y_te)):
        p = model.predict(X)
        out[split] = {**evaluate(y, p), 'PredStd': float(np.std(p))}
    return out

# Pembanding naive (tanpa model)
naive = pd.DataFrame([
    {'Baseline': 'Prediksi 0', 'Split': s, **evaluate(y, np.zeros(len(y)))}
    for s, y in (('Val', y_va), ('Test', y_te))
] + [
    {'Baseline': 'Return kemarin', 'Split': s, **evaluate(y, df_['ret'])}
    for s, y, df_ in (('Val', y_va, final_val), ('Test', y_te, final_test))
])
print(f"{len(MODELS)} model: {list(MODELS)}")
naive

# 7. Ablation Study (metode labelling)
Tidak ada fitur dasar: fitur model hanya hasil labelling (VADER, LM judul, LM isi). Yang dibandingkan adalah kombinasinya:
1 label -> 2 label -> semua label. (Set "tanpa labelling" tidak ada karena model tidak akan punya fitur.)


In [ ]:
from itertools import combinations

# set fitur ablation: {nama: (tahap, daftar kolom)}
ablation_sets = {}
label_names = list(LABEL_GROUPS)
for r in sorted({1, 2, len(label_names)}):
    if r > len(label_names):
        continue
    stage = 'Semua label' if r == len(label_names) else f'{r} label'
    for combo in combinations(label_names, r):
        ablation_sets[' + '.join(combo)] = (stage, [c for g in combo for c in LABEL_GROUPS[g]])

rows = []
for fs_name, (stage, cols) in ablation_sets.items():
    X_tr, X_va, X_te = prepare(cols)
    for m_name, make_model in MODELS.items():
        res = fit_eval(make_model, X_tr, X_va, X_te)
        for split in ('Val', 'Test'):
            rows.append({'Model': m_name, 'Set fitur': fs_name, 'Tahap': stage, 'n_fitur': len(cols),
                         'Split': split, **res[split]})
    print(f'selesai: {fs_name} ({len(cols)} fitur)')

results_ablation = pd.DataFrame(rows)
set_order = list(ablation_sets)
print(f"\n{len(ablation_sets)} set fitur x {len(MODELS)} model")

In [ ]:
# ===== Ringkasan ablation: rata-rata semua model per set fitur =====
summary_abl = (results_ablation.groupby(['Split', 'Set fitur'])[METRICS].mean()
               .reindex(pd.MultiIndex.from_product([['Val', 'Test'], set_order], names=['Split', 'Set fitur'])))
print('Rata-rata metrik semua model per set fitur (fitur = hasil labelling saja):')
display(summary_abl)

# ===== Detail: tiap metrik, model x set fitur =====
for split in ('Val', 'Test'):
    for metric in METRICS:
        pv = (results_ablation[results_ablation['Split'] == split]
              .pivot(index='Model', columns='Set fitur', values=metric)[set_order]
              .reindex(list(MODELS)))
        print(f'\n=== {metric} | {split} ===')
        display(pv)

In [ ]:
# ===== Set fitur terbaik per model (dipilih dari RMSE VALIDATION; test hanya dilaporkan) =====
val_abl  = results_ablation[results_ablation['Split'] == 'Val']
test_abl = results_ablation[results_ablation['Split'] == 'Test'].set_index(['Model', 'Set fitur'])
best_idx = val_abl.loc[val_abl.groupby('Model')['RMSE'].idxmin()]

best_abl = best_idx.set_index('Model')[['Set fitur', 'n_fitur', 'RMSE']].rename(columns={'RMSE': 'Val RMSE'})
for metric in METRICS:
    best_abl[f'Test {metric}'] = [test_abl.loc[(m, s), metric] for m, s in zip(best_abl.index, best_abl['Set fitur'])]
best_abl = best_abl.reindex(list(MODELS)).sort_values('Val RMSE')
print('Set fitur terbaik per model:')
display(best_abl)

# 8. Feature Selection (top-K)
Untuk tiap model, kandidat fitur = **set fitur terbaik dari ablation** (menurut Val RMSE).
Ranking fitur = |korelasi Pearson| dengan target, dihitung di **train saja**. Setelah dipilih, fitur dinormalisasi (scaler fit di train),
model dilatih ulang (parameter default) lalu dievaluasi di val dan test. Hasilnya dibandingkan dengan model tanpa seleksi (semua fitur set terbaik).

Total fitur labelling hanya 16 (VADER 6, LM judul 5, LM isi 5), jadi K default diturunkan ke 5 / 10 / 15.
Top-20 akan identik dengan "Semua fitur". Jika K lebih besar dari jumlah fitur set terbaik, kode otomatis memakai semua fitur.


In [ ]:
TOP_KS = [5, 10, 15]     # total fitur labelling hanya 16; ubah bila perlu
K_LABELS = [f'Top-{k}' for k in TOP_KS] + ['Semua fitur']

def composition(cols):
    cnt = pd.Series([GROUP_OF[c] for c in cols]).value_counts()
    return ' | '.join(f'{g} {n}' for g, n in cnt.items())

rows_fs = []
for m_name, make_model in MODELS.items():
    fs_name = best_abl.loc[m_name, 'Set fitur']
    pool = ablation_sets[fs_name][1]
    # ranking di TRAIN saja (val/test tidak dipakai); fitur konstan -> korelasi 0
    rank = final_train[pool].corrwith(y_tr).abs().fillna(0.0).sort_values(ascending=False).index.tolist()

    for k in TOP_KS:
        sel = rank[:min(k, len(pool))]
        X_tr, X_va, X_te = prepare(sel)
        res = fit_eval(make_model, X_tr, X_va, X_te)
        for split in ('Val', 'Test'):
            rows_fs.append({'Model': m_name, 'Set fitur': fs_name, 'K': f'Top-{k}', 'n_fitur': len(sel),
                            'Split': split, **res[split], 'Komposisi': composition(sel)})

    # pembanding: tanpa seleksi (hasil dari ablation)
    for split in ('Val', 'Test'):
        r = results_ablation[(results_ablation['Model'] == m_name) & (results_ablation['Set fitur'] == fs_name)
                             & (results_ablation['Split'] == split)].iloc[0]
        rows_fs.append({'Model': m_name, 'Set fitur': fs_name, 'K': 'Semua fitur', 'n_fitur': len(pool),
                        'Split': split, **{m: r[m] for m in METRICS}, 'PredStd': np.nan,
                        'Komposisi': composition(pool)})
    print('selesai:', m_name)

results_fs = pd.DataFrame(rows_fs)

In [ ]:
# ===== Perbandingan K: tiap metrik, model x K =====
for split in ('Val', 'Test'):
    for metric in METRICS:
        pv = (results_fs[results_fs['Split'] == split]
              .pivot(index='Model', columns='K', values=metric)[K_LABELS]
              .reindex(list(MODELS)))
        print(f'\n=== {metric} | {split} ===')
        display(pv)

In [ ]:
# ===== K terbaik per model (dipilih dari Val RMSE) + komposisi fitur yang terpilih =====
val_fs  = results_fs[results_fs['Split'] == 'Val']
test_fs = results_fs[results_fs['Split'] == 'Test'].set_index(['Model', 'K'])
best_k  = val_fs.loc[val_fs.groupby('Model')['RMSE'].idxmin()].set_index('Model')

tbl = pd.DataFrame({'Set fitur': best_k['Set fitur'], 'K terbaik': best_k['K'], 'Val RMSE': best_k['RMSE']})
for metric in METRICS:
    tbl[f'Test {metric}'] = [test_fs.loc[(m, k), metric] for m, k in zip(tbl.index, tbl['K terbaik'])]
tbl['Komposisi fitur'] = best_k['Komposisi']
tbl = tbl.reindex(list(MODELS)).sort_values('Val RMSE')
display(tbl)

# ===== Grafik: Test RMSE per model untuk tiap K =====
import matplotlib.pyplot as plt
pv = (results_fs[results_fs['Split'] == 'Test'].pivot(index='Model', columns='K', values='RMSE')[K_LABELS]
      .reindex(list(MODELS)))
ax = pv.plot(kind='bar', figsize=(12, 4.5), width=0.8)
ax.axhline(naive.query("Baseline == 'Prediksi 0' and Split == 'Test'")['RMSE'].iloc[0],
           color='k', ls='--', lw=1, label='Naive: prediksi 0')
ax.set_ylabel('Test RMSE'); ax.set_title('Test RMSE per model: ' + ' / '.join(f'top-{k}' for k in TOP_KS) + ' vs semua fitur')
ax.legend(ncol=5, fontsize=8); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()